# 01 — Generate customers

**Business objective:** every order needs a customer. We need a realistic customer
base with signup dates spread over time and a geographic distribution across our
retail cities, so downstream analysis (e.g. "Sydney customers vs Melbourne customers")
is meaningful rather than arbitrary.

**What we're generating:** 5,000 synthetic customers with name, email, city, signup
date, and a simple segment (Consumer / Business), using the `Faker` library for
realistic-looking names and emails.

In [1]:
import sys
sys.path.insert(0, '..')
from notebooks import nb_config as cfg

import pandas as pd
import numpy as np
from faker import Faker

rng = np.random.default_rng(cfg.SEED)
fake = Faker()
Faker.seed(cfg.SEED)

print(f"Target customers: {cfg.N_CUSTOMERS}")
print(f"Output path: {cfg.DATA_DIR}")

Target customers: 5000
Output path: /home/claude/project/eaida/data/raw


## Generation logic

Customers are assigned a home city drawn from the same city list our stores use
(so "customer lives in Sydney" and "store is in Sydney" refer to the same places).
Signup dates are spread across a 3-year window with slightly more recent signups,
which is typical for a growing retailer.

In [2]:
cities = sorted(set(c for c, _ in cfg.CITIES))
segments = ["Consumer", "Business"]

signup_start = pd.Timestamp("2023-01-01")
signup_end = pd.Timestamp("2026-06-01")

# --- Signup timing now varies by city and month instead of one shared curve ---
# Why: to test acquisition decline vs retention decline separately (Day 4), Sydney
# needs a GENUINE drop in new signups during the anomaly window, not just fewer
# orders. Every other city keeps a normal, growing signup trend.
months = pd.period_range(signup_start, signup_end, freq="M")
n_months = len(months)
anomaly_month_flags = np.zeros(n_months, dtype=bool)
anomaly_month_flags[-cfg.ANOMALY_MONTHS:] = True  # last N months = anomaly window

base_trend = np.linspace(0.6, 1.4, n_months)  # steady customer-base growth over time, like a real retailer

def monthly_weights_for_city(city: str) -> np.ndarray:
    w = base_trend.copy()
    if city == cfg.ANOMALY_CITY:
        w[anomaly_month_flags] *= (1 - cfg.ANOMALY_SIGNUP_REDUCTION)
    return w / w.sum()

city_month_weights = {city: monthly_weights_for_city(city) for city in cities}
month_starts = months.to_timestamp()
month_lengths = months.days_in_month.values

customer_cities = rng.choice(cities, size=cfg.N_CUSTOMERS)

signup_dates = []
for city in customer_cities:
    month_idx = rng.choice(n_months, p=city_month_weights[city])
    day_offset = int(rng.integers(0, int(month_lengths[month_idx])))
    candidate = month_starts[month_idx] + pd.Timedelta(days=day_offset)
    signup_dates.append(min(candidate, signup_end))  # clip: last partial month can't overshoot signup_end

customers = pd.DataFrame({
    "customer_id": range(1, cfg.N_CUSTOMERS + 1),
    "first_name": [fake.first_name() for _ in range(cfg.N_CUSTOMERS)],
    "last_name": [fake.last_name() for _ in range(cfg.N_CUSTOMERS)],
    "city": customer_cities,
    "segment": rng.choice(segments, size=cfg.N_CUSTOMERS, p=[0.85, 0.15]),
    "signup_date": signup_dates,
})
customers["email"] = (
    customers["first_name"].str.lower() + "." + customers["last_name"].str.lower()
    + customers["customer_id"].astype(str) + "@example.com"
)
customers = customers[["customer_id", "first_name", "last_name", "email", "city", "segment", "signup_date"]]
customers.head()

,customer_id,first_name,last_name,email,city,segment,signup_date
0,1,Danielle,Anderson,danielle.anderson1@example.com,Adelaide,Consumer,2023-03-20
1,2,Angel,Hart,angel.hart2@example.com,Perth,Consumer,2026-05-22
2,3,Joshua,Stone,joshua.stone3@example.com,Melbourne,Consumer,2026-01-23
3,4,Jeffrey,Welch,jeffrey.welch4@example.com,Canberra,Consumer,2023-04-09
4,5,Jill,Vasquez,jill.vasquez5@example.com,Canberra,Consumer,2026-05-21


## Sample output & distribution check

In [3]:
print(customers["city"].value_counts())
print()
print(customers["segment"].value_counts(normalize=True))

city
Adelaide     851
Sydney       844
Canberra     842
Brisbane     837
Perth        818
Melbourne    808
Name: count, dtype: int64

segment
Consumer    0.8552
Business    0.1448
Name: proportion, dtype: float64


## Validation

In [4]:
assert customers["customer_id"].is_unique, "customer_id must be unique"
assert customers.isnull().sum().sum() == 0, "no nulls expected"
assert customers["signup_date"].between(signup_start, signup_end).all()
# New: prove the acquisition-decline signal actually made it into the data
monthly_signups = (
    customers.assign(signup_month=customers["signup_date"].dt.to_period("M"))
    .groupby(["signup_month", "city"]).size().unstack(fill_value=0)
)
sydney_recent = monthly_signups["Sydney"].tail(cfg.ANOMALY_MONTHS + 2)
print("Sydney new signups, last few months (should dip in the final",
      cfg.ANOMALY_MONTHS, "months):")
print(sydney_recent)

print("All validation checks passed.")

Sydney new signups, last few months (should dip in the final 2 months):
signup_month
2026-03    28
2026-04    31
2026-05    17
2026-06    18
Freq: M, Name: Sydney, dtype: int64
All validation checks passed.


In [5]:
out_path = cfg.DATA_DIR / "customers.csv"
customers.to_csv(out_path, index=False)
print(f"Wrote {len(customers):,} rows to {out_path}")

Wrote 5,000 rows to /home/claude/project/eaida/data/raw/customers.csv


## Summary

Generated 5,000 customers with unique IDs, realistic names/emails, a city drawn
from our retail footprint, and signup dates skewed toward the recent past. No
nulls, no duplicate IDs. Saved to `data/raw/customers.csv` for use by the
`orders` notebook.